# GVH Diagonal Cubic 0.3.2.7.3.7.3.3.28.2 — FAST
## Nested Diagonal Dynamic N-Body & Nonrigid Shear Activation Audit

### Mission

Avant tout benchmark de précession de Mercure, formaliser proprement le problème à \(N\) corps dans une architecture **diagonale dynamique imbriquée** :

\[
\boxed{
\text{corps}
\subset
\text{sous-système}
\subset
\text{système parent}
\subset
\text{référentiel global}
}
\]

avec trois objectifs distincts :

1. matérialiser la cinématique imbriquée exacte
   \[
   \mathbf r_k=\mathbf X_k+A_k\mathbf r_{k+1},
   \qquad
   A_k=R_kD_k;
   \]
2. vérifier que la dynamique \(N\)-corps de référence reste indépendante de la décomposition de référentiel ;
3. construire une branche diagonale non rigide où le témoin de cisaillement 3D est non nul.

### Provenance canonique

Upstream exécuté `.3.3.28.1` :

`b9110b719d0c19328f2fd417e06a98d2a23b72f810de3256bc4a21ffbfa0085e`

Taille : `41570` octets.

Cellules : `18`.

Résultats amont conservés :

- fermeture cubique vectorielle sur surface de contraintes ;
- hélice rigide shear-free et cubic-blind ;
- \(c_{14}\Omega^3=0\) pour le proxy relativiste rigide ;
- aucun \(\omega_{10},\omega_{21},u_2\) numérique dérivé ;
- aucune validation physique GVH-DCC.

Références OPEN héritées pertinentes :

\[
\boxed{
\texttt{O-14,\ O-26,\ O-27,\ O-29,\ O-31,\ O-32,\ O-34}
}
\]

Ce notebook ne prétend pas redéfinir leur catalogue global.

### Verrou méthodologique

Une transformation \(A_k=R_kD_k\) peut être utilisée de deux manières :

- **FRAME MODE** : simple décomposition de coordonnées, sans nouvelle physique ;
- **PHYSICAL FIELD MODE** : \(D_k\) devient un champ physique et doit alors être dérivé d'une action/EOM.

`.3.3.28.2` ferme le premier mode et prépare le second. Il ne doit jamais confondre les deux.

In [1]:
from __future__ import annotations
import json, sys, math
from pathlib import Path
import numpy as np
import pandas as pd
import sympy as sp

UPSTREAM = {
    "version": "0.3.2.7.3.7.3.3.28.1",
    "sha256": "b9110b719d0c19328f2fd417e06a98d2a23b72f810de3256bc4a21ffbfa0085e",
    "size_bytes": 41570,
    "cells": 18,
    "local_audit_pass": True,
    "rigid_helix_shear_free": True,
    "rigid_helix_cubic_activated": False,
    "critical_273_kinematic_inputs_derived": False,
    "new_GVH_physics_validated": False,
}

G282_UPSTREAM_PROVENANCE_PASS = all([
    UPSTREAM["size_bytes"] == 41570,
    UPSTREAM["cells"] == 18,
    UPSTREAM["local_audit_pass"],
    UPSTREAM["rigid_helix_shear_free"],
    not UPSTREAM["rigid_helix_cubic_activated"],
    not UPSTREAM["critical_273_kinematic_inputs_derived"],
    not UPSTREAM["new_GVH_physics_validated"],
])

G282_BLOCKED_OPEN_REFS = ["O-14","O-26","O-27","O-29","O-31","O-32","O-34"]

assert G282_UPSTREAM_PROVENANCE_PASS

print("Python =", sys.version.split()[0])
print("NumPy =", np.__version__)
print("SymPy =", sp.__version__)
print("G282_UPSTREAM_PROVENANCE_PASS =", G282_UPSTREAM_PROVENANCE_PASS)
print("UPSTREAM_SHA256 =", UPSTREAM["sha256"])
print("G282_BLOCKED_OPEN_REFS =", G282_BLOCKED_OPEN_REFS)

Python = 3.13.15
NumPy = 2.1.3
SymPy = 1.14.0
G282_UPSTREAM_PROVENANCE_PASS = True
UPSTREAM_SHA256 = b9110b719d0c19328f2fd417e06a98d2a23b72f810de3256bc4a21ffbfa0085e
G282_BLOCKED_OPEN_REFS = ['O-14', 'O-26', 'O-27', 'O-29', 'O-31', 'O-32', 'O-34']


# Niveau 1 — GVH

## 1.1 Carte diagonale dynamique imbriquée

À chaque niveau \(k\), on définit :

\[
\boxed{
\mathbf r_k
=
\mathbf X_k(t)+A_k(t)\mathbf r_{k+1}
}
\]

avec :

\[
\boxed{
A_k(t)=R_k(t)D_k(t)
}
\]

où :

\[
R_k(t)\in SO(3),
\qquad
D_k(t)=
\operatorname{diag}
\left(d_{kx},d_{ky},d_{kz}\right).
\]

La composition pour deux niveaux est :

\[
\mathbf r_0
=
\mathbf X_0+
A_0
\left(
\mathbf X_1+A_1\mathbf q
\right).
\]

Les dérivées exactes d'un niveau sont :

\[
\boxed{
\mathbf v
=
\dot{\mathbf X}
+
\dot A\,\mathbf q
+
A\,\dot{\mathbf q}
}
\]

et :

\[
\boxed{
\mathbf a
=
\ddot{\mathbf X}
+
\ddot A\,\mathbf q
+
2\dot A\,\dot{\mathbf q}
+
A\,\ddot{\mathbf q}.
}
\]

L'équation inverse est donc :

\[
\boxed{
\ddot{\mathbf q}
=
A^{-1}
\left[
\mathbf a
-\ddot{\mathbf X}
-\ddot A\,\mathbf q
-2\dot A\,\dot{\mathbf q}
\right].
}
\]

Ces relations sont cinématiques et exactes ; aucune force GVH nouvelle n'est introduite ici.

In [2]:
# Exact numerical forward/reverse audit for a two-level nested frame.

def skew_z(w):
    return np.array([[0.0,-w,0.0],[w,0.0,0.0],[0.0,0.0,0.0]])

def Rz(theta):
    c,s = np.cos(theta), np.sin(theta)
    return np.array([[c,-s,0.0],[s,c,0.0],[0.0,0.0,1.0]])

def frame_state(t, w, h, X0, V0, A0):
    h = np.asarray(h, dtype=float)
    H = np.diag(h)
    R = Rz(w*t)
    W = skew_z(w)
    D = np.diag(np.exp(h*t))
    Dd = D @ H
    Ddd = D @ H @ H

    Rd = W @ R
    Rdd = W @ W @ R

    A = R @ D
    Ad = Rd @ D + R @ Dd
    Add = Rdd @ D + 2*Rd @ Dd + R @ Ddd

    X = np.asarray(X0) + np.asarray(V0)*t + 0.5*np.asarray(A0)*t*t
    Xd = np.asarray(V0) + np.asarray(A0)*t
    Xdd = np.asarray(A0)
    return dict(X=X, Xd=Xd, Xdd=Xdd, A=A, Ad=Ad, Add=Add)

def forward_one(child, F):
    r,v,a = child
    rp = F["X"] + F["A"] @ r
    vp = F["Xd"] + F["Ad"] @ r + F["A"] @ v
    ap = F["Xdd"] + F["Add"] @ r + 2*F["Ad"] @ v + F["A"] @ a
    return rp,vp,ap

def reverse_one(parent, F):
    rp,vp,ap = parent
    r = np.linalg.solve(F["A"], rp - F["X"])
    v = np.linalg.solve(F["A"], vp - F["Xd"] - F["Ad"] @ r)
    a = np.linalg.solve(F["A"], ap - F["Xdd"] - F["Add"] @ r - 2*F["Ad"] @ v)
    return r,v,a

t0 = 0.73
F0 = frame_state(
    t0, w=0.31, h=[0.020,-0.010,-0.010],
    X0=[1.0,-2.0,0.5], V0=[0.2,0.05,-0.1], A0=[0.01,-0.02,0.005]
)
F1 = frame_state(
    t0, w=-0.17, h=[0.030,-0.015,-0.015],
    X0=[-0.4,0.7,0.2], V0=[-0.03,0.04,0.02], A0=[0.002,0.001,-0.003]
)

leaf = (
    np.array([0.8,-0.3,0.5]),
    np.array([0.04,0.07,-0.02]),
    np.array([-0.01,0.005,0.003]),
)

level1 = forward_one(leaf, F1)
global_state = forward_one(level1, F0)

recover1 = reverse_one(global_state, F0)
recover_leaf = reverse_one(recover1, F1)

errors = {
    "r": float(np.max(np.abs(recover_leaf[0]-leaf[0]))),
    "v": float(np.max(np.abs(recover_leaf[1]-leaf[1]))),
    "a": float(np.max(np.abs(recover_leaf[2]-leaf[2]))),
}
G282_NESTED_FORWARD_REVERSE_TOL = 1e-12
G282_NESTED_FORWARD_REVERSE_PASS = max(errors.values()) < G282_NESTED_FORWARD_REVERSE_TOL

assert G282_NESTED_FORWARD_REVERSE_PASS

print("nested recovery errors =", errors)
print("tolerance =", G282_NESTED_FORWARD_REVERSE_TOL)
print("G282_NESTED_FORWARD_REVERSE_PASS =", G282_NESTED_FORWARD_REVERSE_PASS)

nested recovery errors = {'r': 1.1102230246251565e-16, 'v': 5.551115123125783e-17, 'a': 1.3877787807814457e-17}
tolerance = 1e-12
G282_NESTED_FORWARD_REVERSE_PASS = True


## 1.2 Témoin de cisaillement diagonal non rigide

Pour :

\[
A=RD,
\]

définissons le gradient cinématique :

\[
\boxed{
B:=\dot A A^{-1}.
}
\]

Avec :

\[
W=\dot R R^{-1},
\qquad
H=\dot D D^{-1},
\]

on obtient :

\[
B=W+RHR^T.
\]

Comme \(W\) est antisymétrique :

\[
\operatorname{sym}(B)=RHR^T.
\]

Le témoin de cisaillement spatial est :

\[
\boxed{
S
=
\operatorname{sym}(B)
-\frac13\operatorname{tr}(B)I
=
R
\left[
H-\frac13\operatorname{tr}(H)I
\right]
R^T.
}
\]

Ainsi :

\[
H_x=H_y=H_z
\Longrightarrow
S=0,
\]

mais :

\[
\boxed{
H_i\neq H_j
\Longrightarrow
S\neq0
}
\]

génériquement.

Important : \(S_{ij}\) est ici un **témoin cinématique 3D**. `.3.3.28.2` ne pose pas encore :

\[
S_{ij}\equiv\sigma_{ij}
\]

comme identité covariante 4D.

In [3]:
# Exact symbolic proof of the 3D diagonal-shear identity.

th,w = sp.symbols("theta w", real=True)
hx,hy,hz = sp.symbols("h_x h_y h_z", real=True)

R = sp.Matrix([
    [sp.cos(th),-sp.sin(th),0],
    [sp.sin(th), sp.cos(th),0],
    [0,0,1],
])
W = sp.Matrix([[0,-w,0],[w,0,0],[0,0,0]])
H = sp.diag(hx,hy,hz)

B = W + R*H*R.T
SymB = sp.simplify((B+B.T)/2)
Hiso = sp.simplify((hx+hy+hz)/3)
S = sp.simplify(SymB - Hiso*sp.eye(3))
S_expected = sp.simplify(R*(H-Hiso*sp.eye(3))*R.T)

G282_3D_SHEAR_IDENTITY_EXACT_PASS = all(
    sp.simplify(S[i,j]-S_expected[i,j]) == 0
    for i in range(3) for j in range(3)
)

S_iso = sp.simplify(S.subs({hy:hx,hz:hx}))
G282_ISOTROPIC_DIAGONAL_SHEAR_ZERO_PASS = all(
    sp.simplify(S_iso[i,j]) == 0
    for i in range(3) for j in range(3)
)

# Numeric anisotropic witness, volume-preserving h_x+h_y+h_z=0.
S_num = np.array(S.subs({
    th:0.63, hx:0.03, hy:-0.01, hz:-0.02
}).evalf(), dtype=float)
shear_norm2 = float(np.sum(S_num*S_num))

G282_NONRIGID_DIAGONAL_3D_SHEAR_ACTIVATED = shear_norm2 > 1e-10
G282_COVARIANT_SIGMA_IDENTIFICATION_PROVEN = False

assert G282_3D_SHEAR_IDENTITY_EXACT_PASS
assert G282_ISOTROPIC_DIAGONAL_SHEAR_ZERO_PASS
assert G282_NONRIGID_DIAGONAL_3D_SHEAR_ACTIVATED
assert not G282_COVARIANT_SIGMA_IDENTIFICATION_PROVEN

print("G282_3D_SHEAR_IDENTITY_EXACT_PASS =", G282_3D_SHEAR_IDENTITY_EXACT_PASS)
print("G282_ISOTROPIC_DIAGONAL_SHEAR_ZERO_PASS =", G282_ISOTROPIC_DIAGONAL_SHEAR_ZERO_PASS)
print("S_num =")
print(S_num)
print("||S||_F^2 =", shear_norm2)
print("G282_NONRIGID_DIAGONAL_3D_SHEAR_ACTIVATED =",
      G282_NONRIGID_DIAGONAL_3D_SHEAR_ACTIVATED)
print("G282_COVARIANT_SIGMA_IDENTIFICATION_PROVEN =",
      G282_COVARIANT_SIGMA_IDENTIFICATION_PROVEN)

G282_3D_SHEAR_IDENTITY_EXACT_PASS = True
G282_ISOTROPIC_DIAGONAL_SHEAR_ZERO_PASS = True
S_num =
[[ 0.01611634  0.01904181  0.        ]
 [ 0.01904181  0.00388366  0.        ]
 [ 0.          0.         -0.02      ]]
||S||_F^2 = 0.0014000000000000002
G282_NONRIGID_DIAGONAL_3D_SHEAR_ACTIVATED = True
G282_COVARIANT_SIGMA_IDENTIFICATION_PROVEN = False


## 1.3 Dynamique \(N\)-corps de référence

Le benchmark physique minimal est volontairement conservateur :

\[
\boxed{
\ddot{\mathbf r}_i
=
-G
\sum_{j\neq i}
m_j
\frac{
\mathbf r_i-\mathbf r_j
}{
|\mathbf r_i-\mathbf r_j|^3
}.
}
\]

Le rôle de `.3.3.28.2` est de vérifier que cette dynamique peut être exprimée dans les référentiels imbriqués sans modifier la trajectoire globale.

La formulation générale à \(N\) corps est donc matérialisée, mais on **ne prétend pas** obtenir une solution analytique fermée générale pour \(N\ge3\).

Les invariants de contrôle sont :

\[
\mathbf P=\sum_i m_i\mathbf v_i,
\]

\[
\mathbf L=\sum_i\mathbf r_i\times m_i\mathbf v_i,
\]

\[
E=
\frac12\sum_i m_i v_i^2
-
G\sum_{i<j}\frac{m_im_j}{r_{ij}}.
\]

Le caractère lié/non lié reste un diagnostic construit à partir de la dynamique, pas une nouvelle loi de force GVH.

In [4]:
# Generic N-body pairwise acceleration and covariance audit.

G = 1.0

def nbody_acceleration(r, m, G=1.0):
    r = np.asarray(r, dtype=float)
    m = np.asarray(m, dtype=float)
    N = len(m)
    a = np.zeros_like(r)
    for i in range(N):
        for j in range(i+1,N):
            d = r[j]-r[i]
            dist = np.linalg.norm(d)
            if dist <= 0:
                raise ValueError("collision singularity")
            base = G*d/dist**3
            a[i] += m[j]*base
            a[j] -= m[i]*base
    return a

rng = np.random.default_rng(282)
Nw = 8
m8 = rng.uniform(0.5,2.0,size=Nw)
r8 = rng.normal(size=(Nw,3))
r8 += np.arange(Nw)[:,None]*np.array([0.7,0.2,-0.1])
a8 = nbody_acceleration(r8,m8,G=G)

force_balance = np.sum(m8[:,None]*a8,axis=0)
force_scale = np.sum(np.linalg.norm(m8[:,None]*a8,axis=1))
force_rel = float(np.linalg.norm(force_balance)/force_scale)

shift = np.array([12.3,-7.4,3.1])
a_shift = nbody_acceleration(r8+shift,m8,G=G)
translation_err = float(np.max(np.abs(a_shift-a8)))

Q = Rz(0.91) @ np.array([
    [1.0,0.0,0.0],
    [0.0,np.cos(0.37),-np.sin(0.37)],
    [0.0,np.sin(0.37), np.cos(0.37)],
])
a_rot = nbody_acceleration((Q@r8.T).T,m8,G=G)
rotation_err = float(np.max(np.abs(a_rot-(Q@a8.T).T)))

G282_NBODY_INTERNAL_FORCE_BALANCE_PASS = force_rel < 1e-14
G282_NBODY_TRANSLATION_COVARIANCE_PASS = translation_err < 1e-12
G282_NBODY_ROTATION_COVARIANCE_PASS = rotation_err < 1e-12
G282_GENERIC_NBODY_FORCE_LAW_MATERIALIZED = True

assert G282_NBODY_INTERNAL_FORCE_BALANCE_PASS
assert G282_NBODY_TRANSLATION_COVARIANCE_PASS
assert G282_NBODY_ROTATION_COVARIANCE_PASS

print("N =", Nw)
print("relative internal-force balance =", force_rel)
print("translation covariance max error =", translation_err)
print("rotation covariance max error =", rotation_err)
print("G282_NBODY_INTERNAL_FORCE_BALANCE_PASS =", G282_NBODY_INTERNAL_FORCE_BALANCE_PASS)
print("G282_NBODY_TRANSLATION_COVARIANCE_PASS =", G282_NBODY_TRANSLATION_COVARIANCE_PASS)
print("G282_NBODY_ROTATION_COVARIANCE_PASS =", G282_NBODY_ROTATION_COVARIANCE_PASS)

N = 8
relative internal-force balance = 5.93825722400258e-17
translation covariance max error = 1.1102230246251565e-15
rotation covariance max error = 6.661338147750939e-16
G282_NBODY_INTERNAL_FORCE_BALANCE_PASS = True
G282_NBODY_TRANSLATION_COVARIANCE_PASS = True
G282_NBODY_ROTATION_COVARIANCE_PASS = True


# Niveau 2 — Physique établie comme benchmark

## 2.1 Deux corps : orbite circulaire de Kepler

On utilise \(G=1\), deux masses égales :

\[
m_1=m_2=1,
\]

séparées de :

\[
d=1.
\]

Chaque corps se trouve à :

\[
r=\frac12
\]

du barycentre. La condition circulaire donne :

\[
\frac{v^2}{1/2}=1
\Longrightarrow
\boxed{
v=\frac{1}{\sqrt2}.
}
\]

Le test numérique n'est pas une prédiction GVH. Il vérifie que l'architecture proposée peut reproduire une dynamique de référence sans dégrader ses invariants.

## 2.2 Trois corps : solution équilatérale de Lagrange

Pour trois masses égales \(m=1\) aux sommets d'un triangle équilatéral de côté \(1\) :

\[
R=\frac{1}{\sqrt3},
\qquad
\omega=\sqrt3,
\qquad
v=1.
\]

Ce benchmark introduit déjà une dynamique réellement \(N=3\), sans prétendre résoudre analytiquement le problème général des trois corps.

In [5]:
def total_energy(r,v,m,G=1.0):
    T = 0.5*np.sum(m[:,None]*v*v)
    U = 0.0
    for i in range(len(m)):
        for j in range(i+1,len(m)):
            U -= G*m[i]*m[j]/np.linalg.norm(r[j]-r[i])
    return T+U

def total_L(r,v,m):
    return np.sum(np.cross(r,m[:,None]*v),axis=0)

def velocity_verlet(r,v,m,dt,steps,G=1.0,shape_reference=None):
    r = np.array(r,dtype=float,copy=True)
    v = np.array(v,dtype=float,copy=True)
    a = nbody_acceleration(r,m,G)
    E0 = total_energy(r,v,m,G)
    L0 = total_L(r,v,m)
    max_E_rel = 0.0
    max_L_rel = 0.0
    max_shape = 0.0
    for _ in range(steps):
        r = r + v*dt + 0.5*a*dt*dt
        anew = nbody_acceleration(r,m,G)
        v = v + 0.5*(a+anew)*dt
        a = anew

        max_E_rel = max(max_E_rel,abs((total_energy(r,v,m,G)-E0)/E0))
        Lden = max(np.linalg.norm(L0),1e-30)
        max_L_rel = max(max_L_rel,np.linalg.norm(total_L(r,v,m)-L0)/Lden)

        if shape_reference is not None:
            ds = []
            for i in range(len(m)):
                for j in range(i+1,len(m)):
                    ds.append(np.linalg.norm(r[j]-r[i]))
            max_shape = max(max_shape,float(np.max(np.abs(np.asarray(ds)-shape_reference))))
    return r,v,max_E_rel,max_L_rel,max_shape

# N=2 circular benchmark.
m2 = np.array([1.0,1.0])
r2 = np.array([[-0.5,0,0],[0.5,0,0]],dtype=float)
vc = 1/math.sqrt(2)
v2 = np.array([[0,-vc,0],[0,vc,0]],dtype=float)
T2 = 2*math.pi*0.5/vc
dt2 = T2/2000
_,_,E2,L2,_ = velocity_verlet(r2,v2,m2,dt2,10000,G=1.0)

# N=3 Lagrange equilateral benchmark.
m3 = np.ones(3)
R3 = 1/math.sqrt(3)
ang = np.array([0,2*math.pi/3,4*math.pi/3])
r3 = np.c_[R3*np.cos(ang),R3*np.sin(ang),np.zeros(3)]
w3 = math.sqrt(3)
v3 = np.c_[-w3*R3*np.sin(ang),w3*R3*np.cos(ang),np.zeros(3)]
T3 = 2*math.pi/w3
dt3 = T3/2000
_,_,E3,L3,shape3 = velocity_verlet(
    r3,v3,m3,dt3,10000,G=1.0,shape_reference=np.ones(3)
)

G282_TWO_BODY_ENERGY_CONSERVATION_PASS = E2 < 1e-8
G282_TWO_BODY_ANGULAR_MOMENTUM_PASS = L2 < 1e-12
G282_THREE_BODY_LAGRANGE_ENERGY_PASS = E3 < 1e-8
G282_THREE_BODY_LAGRANGE_ANGULAR_MOMENTUM_PASS = L3 < 1e-12
G282_THREE_BODY_LAGRANGE_SHAPE_PASS = shape3 < 1e-4

G282_ESTABLISHED_NBODY_BENCHMARK_PASS = all([
    G282_TWO_BODY_ENERGY_CONSERVATION_PASS,
    G282_TWO_BODY_ANGULAR_MOMENTUM_PASS,
    G282_THREE_BODY_LAGRANGE_ENERGY_PASS,
    G282_THREE_BODY_LAGRANGE_ANGULAR_MOMENTUM_PASS,
    G282_THREE_BODY_LAGRANGE_SHAPE_PASS,
])

assert G282_ESTABLISHED_NBODY_BENCHMARK_PASS

print("N=2 max relative energy drift =", E2)
print("N=2 max relative angular-momentum drift =", L2)
print("N=3 max relative energy drift =", E3)
print("N=3 max relative angular-momentum drift =", L3)
print("N=3 max pair-distance error =", shape3)
print("G282_ESTABLISHED_NBODY_BENCHMARK_PASS =", G282_ESTABLISHED_NBODY_BENCHMARK_PASS)

N=2 max relative energy drift = 2.4355628625016828e-11
N=2 max relative angular-momentum drift = 7.379434555813744e-15
N=3 max relative energy drift = 2.4355628625016834e-11
N=3 max relative angular-momentum drift = 4.615110447320553e-15
N=3 max pair-distance error = 5.2564212276173805e-06
G282_ESTABLISHED_NBODY_BENCHMARK_PASS = True


## 2.3 Hiérarchie barycentrique imbriquée

Pour un groupe \(G_\alpha\), on définit :

\[
M_\alpha=\sum_{i\in G_\alpha}m_i,
\]

\[
\boxed{
\mathbf R_\alpha
=
\frac{1}{M_\alpha}
\sum_{i\in G_\alpha}
m_i\mathbf r_i.
}
\]

Les coordonnées locales diagonales sont :

\[
\boxed{
\mathbf q_i
=
A_\alpha^{-1}
(\mathbf r_i-\mathbf R_\alpha).
}
\]

Par construction :

\[
\sum_{i\in G_\alpha}m_i\mathbf q_i=0.
\]

Cette décomposition permet par exemple :

\[
\text{planète}
\subset
\text{système planétaire}
\subset
\text{système solaire}
\subset
\text{Galaxie}.
\]

Mais le choix du groupement ne doit pas modifier les positions globales ni la force \(N\)-corps lorsqu'il est utilisé comme simple référentiel.

In [6]:
# Barycentric hierarchical decomposition audit on the N=8 snapshot.

groups = [np.array([0,1,2,3]), np.array([4,5,6,7])]
Agrp = frame_state(
    0.41, w=0.22, h=[0.015,-0.005,-0.010],
    X0=[0,0,0], V0=[0,0,0], A0=[0,0,0]
)["A"]

reconstruction_errors = []
mass_center_local_errors = []

for ids in groups:
    mg = m8[ids]
    rg = r8[ids]
    Mtot = np.sum(mg)
    Rcm = np.sum(mg[:,None]*rg,axis=0)/Mtot
    q = np.array([np.linalg.solve(Agrp,ri-Rcm) for ri in rg])
    rrec = Rcm + (Agrp@q.T).T

    reconstruction_errors.append(float(np.max(np.abs(rrec-rg))))
    mass_center_local_errors.append(
        float(np.linalg.norm(np.sum(mg[:,None]*q,axis=0))/Mtot)
    )

G282_HIERARCHICAL_POSITION_RECONSTRUCTION_PASS = max(reconstruction_errors) < 1e-12
G282_LOCAL_BARYCENTER_ZERO_PASS = max(mass_center_local_errors) < 1e-12

assert G282_HIERARCHICAL_POSITION_RECONSTRUCTION_PASS
assert G282_LOCAL_BARYCENTER_ZERO_PASS

print("group reconstruction errors =", reconstruction_errors)
print("local barycenter errors =", mass_center_local_errors)
print("G282_HIERARCHICAL_POSITION_RECONSTRUCTION_PASS =",
      G282_HIERARCHICAL_POSITION_RECONSTRUCTION_PASS)
print("G282_LOCAL_BARYCENTER_ZERO_PASS =", G282_LOCAL_BARYCENTER_ZERO_PASS)

group reconstruction errors = [1.1102230246251565e-16, 2.220446049250313e-16]
local barycenter errors = [1.2665309286330457e-16, 1.6712174031704211e-16]
G282_HIERARCHICAL_POSITION_RECONSTRUCTION_PASS = True
G282_LOCAL_BARYCENTER_ZERO_PASS = True


# Niveau 3 — Équation locale dans le référentiel diagonal dynamique

Si la force physique est calculée globalement :

\[
\mathbf a_i^{\rm phys}
=
-G\sum_{j\neq i}m_j
\frac{\mathbf r_i-\mathbf r_j}{r_{ij}^3},
\]

alors dans un référentiel :

\[
\mathbf r_i=\mathbf X+A\mathbf q_i,
\]

on obtient :

\[
\boxed{
\ddot{\mathbf q}_i
=
A^{-1}
\left[
\mathbf a_i^{\rm phys}
-\ddot{\mathbf X}
-\ddot A\,\mathbf q_i
-2\dot A\,\dot{\mathbf q}_i
\right].
}
\]

Les termes :

\[
-\ddot{\mathbf X},
\qquad
-\ddot A\,\mathbf q,
\qquad
-2\dot A\,\dot{\mathbf q}
\]

sont des termes de référentiel.

Ils ne doivent pas être réinterprétés comme une nouvelle interaction GVH tant que \(A(t)\), et en particulier \(D(t)\), n'est pas dérivé d'une EOM physique.

In [7]:
# Direct global-vs-local acceleration reconstruction for the N=8 physical snapshot.

rng2 = np.random.default_rng(8282)
v8 = rng2.normal(scale=0.15,size=(Nw,3))

F = frame_state(
    0.52, w=0.27, h=[0.025,-0.010,-0.015],
    X0=[0.3,-0.4,0.2], V0=[0.04,-0.03,0.01], A0=[0.002,-0.001,0.003]
)

q8 = np.array([np.linalg.solve(F["A"],ri-F["X"]) for ri in r8])
qd8 = np.array([
    np.linalg.solve(F["A"],vi-F["Xd"]-F["Ad"]@qi)
    for vi,qi in zip(v8,q8)
])

a_global = nbody_acceleration(r8,m8,G=G)

qdd8 = np.array([
    np.linalg.solve(
        F["A"],
        ai-F["Xdd"]-F["Add"]@qi-2*F["Ad"]@qdi
    )
    for ai,qi,qdi in zip(a_global,q8,qd8)
])

a_reconstructed = np.array([
    F["Xdd"]+F["Add"]@qi+2*F["Ad"]@qdi+F["A"]@qddi
    for qi,qdi,qddi in zip(q8,qd8,qdd8)
])

local_global_acc_error = float(np.max(np.abs(a_reconstructed-a_global)))

G282_LOCAL_GLOBAL_ACCELERATION_EQUIVALENCE_PASS = local_global_acc_error < 1e-12
G282_FRAME_TERMS_EXPLICIT = True
G282_FRAME_TERMS_NEW_FORCE_INTERPRETATION_AUTHORIZED = False

assert G282_LOCAL_GLOBAL_ACCELERATION_EQUIVALENCE_PASS
assert G282_FRAME_TERMS_EXPLICIT
assert not G282_FRAME_TERMS_NEW_FORCE_INTERPRETATION_AUTHORIZED

print("global/local reconstructed acceleration max error =", local_global_acc_error)
print("G282_LOCAL_GLOBAL_ACCELERATION_EQUIVALENCE_PASS =",
      G282_LOCAL_GLOBAL_ACCELERATION_EQUIVALENCE_PASS)
print("G282_FRAME_TERMS_EXPLICIT =", G282_FRAME_TERMS_EXPLICIT)
print("G282_FRAME_TERMS_NEW_FORCE_INTERPRETATION_AUTHORIZED =",
      G282_FRAME_TERMS_NEW_FORCE_INTERPRETATION_AUTHORIZED)

global/local reconstructed acceleration max error = 2.220446049250313e-16
G282_LOCAL_GLOBAL_ACCELERATION_EQUIVALENCE_PASS = True
G282_FRAME_TERMS_EXPLICIT = True
G282_FRAME_TERMS_NEW_FORCE_INTERPRETATION_AUTHORIZED = False


## 3.1 Audit Prescribed / Derived

Le résultat positif de `.3.3.28.2` est structurel :

\[
\boxed{
\text{la dynamique }N\text{-corps peut être portée exactement par une hiérarchie }A_k=R_kD_k.
}
\]

Mais dans ce notebook :

\[
R_k(t),D_k(t),\mathbf X_k(t)
\]

sont des **fonctions de référentiel prescrites**.

Le témoin :

\[
S_{ij}\neq0
\]

montre qu'une déformation diagonale non rigide peut produire un cisaillement spatial cinématique.

Il ne démontre pas encore que :

\[
D_k(t)
\]

est sélectionné par l'action GVH, ni que :

\[
S_{ij}
\]

est exactement la projection spatiale du tenseur covariant :

\[
\sigma_{\mu\nu}.
\]

En conséquence :

\[
\boxed{
\texttt{NEW\_GVH\_NBODY\_FORCE\_LAW\_DERIVED=False}
}
\]

et :

\[
\boxed{
\texttt{PHYSICAL\_DIAGONAL\_FIELD\_EOM\_DERIVED=False}.
}
\]

In [8]:
pd_ledger = pd.DataFrame([
    {
        "object":"global N-body positions r_i",
        "role":"DYNAMICAL_REFERENCE_VARIABLES",
        "status":"DERIVED_BY_REFERENCE_FORCE_LAW_NUMERICALLY",
        "physical_GVH_derived":False,
    },
    {
        "object":"X_k(t)",
        "role":"NESTED_FRAME_ORIGIN",
        "status":"PRESCRIBED_FRAME_DATA",
        "physical_GVH_derived":False,
    },
    {
        "object":"R_k(t)",
        "role":"NESTED_FRAME_ROTATION",
        "status":"PRESCRIBED_FRAME_DATA",
        "physical_GVH_derived":False,
    },
    {
        "object":"D_k(t)",
        "role":"NESTED_DIAGONAL_DEFORMATION",
        "status":"PRESCRIBED_FRAME_DATA",
        "physical_GVH_derived":False,
    },
    {
        "object":"S_ij",
        "role":"3D_KINEMATIC_SHEAR_WITNESS",
        "status":"DERIVED_FROM_PRESCRIBED_D_k",
        "physical_GVH_derived":False,
    },
    {
        "object":"sigma_mn",
        "role":"COVARIANT_GVH_SHEAR",
        "status":"NOT_IDENTIFIED_WITH_S_ij",
        "physical_GVH_derived":False,
    },
    {
        "object":"Newtonian pair force",
        "role":"ESTABLISHED_REFERENCE_DYNAMICS",
        "status":"BENCHMARK_NOT_NEW_GVH_FORCE",
        "physical_GVH_derived":False,
    },
    {
        "object":"bound/unbound label",
        "role":"DIAGNOSTIC",
        "status":"DERIVED_CLASSIFICATION_NOT_FORCE_LAW",
        "physical_GVH_derived":False,
    },
])

G282_NBODY_STRUCTURAL_CLOSURE_ADVANCE = True
G282_NONRIGID_SHEAR_KINEMATIC_ACTIVATION = G282_NONRIGID_DIAGONAL_3D_SHEAR_ACTIVATED
G282_PHYSICAL_DIAGONAL_FIELD_EOM_DERIVED = False
G282_NEW_GVH_NBODY_FORCE_LAW_DERIVED = False
G282_GENERAL_NBODY_ANALYTIC_SOLUTION_CLAIMED = False
G282_MERCURY_GVH_PRECESSION_PREDICTION_AUTHORIZED = False

assert G282_NBODY_STRUCTURAL_CLOSURE_ADVANCE
assert G282_NONRIGID_SHEAR_KINEMATIC_ACTIVATION
assert not G282_PHYSICAL_DIAGONAL_FIELD_EOM_DERIVED
assert not G282_NEW_GVH_NBODY_FORCE_LAW_DERIVED
assert not G282_GENERAL_NBODY_ANALYTIC_SOLUTION_CLAIMED
assert not G282_MERCURY_GVH_PRECESSION_PREDICTION_AUTHORIZED

print(pd_ledger.to_string(index=False))
print("G282_NBODY_STRUCTURAL_CLOSURE_ADVANCE =", G282_NBODY_STRUCTURAL_CLOSURE_ADVANCE)
print("G282_NONRIGID_SHEAR_KINEMATIC_ACTIVATION =", G282_NONRIGID_SHEAR_KINEMATIC_ACTIVATION)
print("G282_PHYSICAL_DIAGONAL_FIELD_EOM_DERIVED =", G282_PHYSICAL_DIAGONAL_FIELD_EOM_DERIVED)
print("G282_NEW_GVH_NBODY_FORCE_LAW_DERIVED =", G282_NEW_GVH_NBODY_FORCE_LAW_DERIVED)
print("G282_MERCURY_GVH_PRECESSION_PREDICTION_AUTHORIZED =",
      G282_MERCURY_GVH_PRECESSION_PREDICTION_AUTHORIZED)

                     object                           role                                     status  physical_GVH_derived
global N-body positions r_i  DYNAMICAL_REFERENCE_VARIABLES DERIVED_BY_REFERENCE_FORCE_LAW_NUMERICALLY                 False
                     X_k(t)            NESTED_FRAME_ORIGIN                      PRESCRIBED_FRAME_DATA                 False
                     R_k(t)          NESTED_FRAME_ROTATION                      PRESCRIBED_FRAME_DATA                 False
                     D_k(t)    NESTED_DIAGONAL_DEFORMATION                      PRESCRIBED_FRAME_DATA                 False
                       S_ij     3D_KINEMATIC_SHEAR_WITNESS                DERIVED_FROM_PRESCRIBED_D_k                 False
                   sigma_mn            COVARIANT_GVH_SHEAR                   NOT_IDENTIFIED_WITH_S_ij                 False
       Newtonian pair force ESTABLISHED_REFERENCE_DYNAMICS                BENCHMARK_NOT_NEW_GVH_FORCE                 False
        

# Niveau 4 — Verdict scientifique

`.3.3.28.2` ferme le **problème de représentation imbriquée** du \(N\)-corps dans le scope testé :

### Résultat A — récursion imbriquée exacte

\[
\boxed{
(\mathbf r,\mathbf v,\mathbf a)
\longleftrightarrow
(\mathbf q,\dot{\mathbf q},\ddot{\mathbf q})
}
\]

est réversible niveau par niveau à précision machine.

### Résultat B — dynamique \(N\)-corps inchangée en FRAME MODE

Les tests :

- balance exacte des forces internes ;
- covariance par translation ;
- covariance par rotation ;
- benchmark Kepler \(N=2\) ;
- benchmark Lagrange \(N=3\) ;
- reconstruction barycentrique imbriquée ;
- équivalence accélération globale/locale ;

passent dans les tolérances numériques fixées.

### Résultat C — branche diagonale non rigide activée

\[
\boxed{
H_x,H_y,H_z\ \text{anisotropes}
\Longrightarrow
S_{ij}\neq0.
}
\]

L'obstacle de `.3.3.28.1` :

\[
\text{rigid helix}\Rightarrow\text{shear-free}
\]

est donc contourné **au niveau cinématique 3D**.

### Ce qui reste bloqué

On ne peut pas encore conclure :

\[
S_{ij}=\sigma_{ij}^{\rm GVH}
\]

en 4D covariant.

On ne peut pas non plus conclure que \(D_k(t)\) est une dynamique physique plutôt qu'un référentiel choisi.

Ainsi :

\[
\boxed{
\texttt{NEW\_GVH\_PHYSICS\_VALIDATED=False}
}
\]

et une prédiction GVH de la précession de Mercure reste non autorisée.

La prochaine étape doit construire l'embedding covariant du cisaillement diagonal imbriqué et tester si l'action `.3.3.28` sélectionne réellement \(D_k(t)\) ou une relation équivalente.

In [9]:
LEVEL1 = "NESTED_DIAGONAL_NBODY_KINEMATICS_AND_REVERSE_DYNAMICS_MATERIALIZED"
LEVEL2 = "NEWTONIAN_NBODY_COVARIANCE_KEPLER_AND_LAGRANGE_BENCHMARKS_PASS"
LEVEL3 = "NUMERIC_NESTED_RECONSTRUCTION_CONSERVATION_AND_SHEAR_WITNESSES"
LEVEL4 = "BLOCKED_PHYSICAL_DIAGONAL_FIELD_EOM_AND_COVARIANT_SHEAR_IDENTIFICATION"

G282_FULL_NESTED_FRAME_NBODY_ARCHITECTURE_MATERIALIZED = all([
    G282_NESTED_FORWARD_REVERSE_PASS,
    G282_GENERIC_NBODY_FORCE_LAW_MATERIALIZED,
    G282_HIERARCHICAL_POSITION_RECONSTRUCTION_PASS,
    G282_LOCAL_GLOBAL_ACCELERATION_EQUIVALENCE_PASS,
])

G282_REFERENCE_PHYSICS_BENCHMARKS_PASS = all([
    G282_NBODY_INTERNAL_FORCE_BALANCE_PASS,
    G282_NBODY_TRANSLATION_COVARIANCE_PASS,
    G282_NBODY_ROTATION_COVARIANCE_PASS,
    G282_ESTABLISHED_NBODY_BENCHMARK_PASS,
])

G282_NEW_GVH_PHYSICS_VALIDATED = False
G282_REAL_DATA_READY = False

G282_LOCAL_AUDIT_PASS = all([
    G282_UPSTREAM_PROVENANCE_PASS,
    G282_FULL_NESTED_FRAME_NBODY_ARCHITECTURE_MATERIALIZED,
    G282_REFERENCE_PHYSICS_BENCHMARKS_PASS,
    G282_3D_SHEAR_IDENTITY_EXACT_PASS,
    G282_ISOTROPIC_DIAGONAL_SHEAR_ZERO_PASS,
    G282_NONRIGID_DIAGONAL_3D_SHEAR_ACTIVATED,
    G282_LOCAL_BARYCENTER_ZERO_PASS,
    G282_NBODY_STRUCTURAL_CLOSURE_ADVANCE,
    not G282_COVARIANT_SIGMA_IDENTIFICATION_PROVEN,
    not G282_PHYSICAL_DIAGONAL_FIELD_EOM_DERIVED,
    not G282_NEW_GVH_NBODY_FORCE_LAW_DERIVED,
    not G282_GENERAL_NBODY_ANALYTIC_SOLUTION_CLAIMED,
    not G282_MERCURY_GVH_PRECESSION_PREDICTION_AUTHORIZED,
    not G282_NEW_GVH_PHYSICS_VALIDATED,
])

G282_OBSTRUCTIONS = [
    "S_ij_IS_ONLY_A_3D_KINEMATIC_SHEAR_WITNESS_NOT_YET_COVARIANT_SIGMA_mn",
    "D_k_OF_t_REMAINS_PRESCRIBED_FRAME_DATA_NOT_ACTION_DERIVED",
    "NO_NEW_GVH_NBODY_FORCE_LAW_DERIVED",
    "CUBIC_METRIC_STRESS_STILL_NOT_FULLY_COMPONENT_EXPANDED_IN_ARBITRARY_4D",
    "TRIAD_OWN_KINETIC_TERM_STILL_ABSENT",
    "MERCURY_GVH_PRECESSION_NOT_YET_AUTHORIZED",
]

G282_NEXT_AUTHORIZED = (
    "0.3.2.7.3.7.3.3.28.3_"
    "COVARIANT_NESTED_SHEAR_EMBEDDING_AND_PHYSICAL_DIAGONAL_FIELD_EOM_GATE_AUDIT"
)

assert G282_LOCAL_AUDIT_PASS

artifact = {
    "notebook": (
        "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28.2_"
        "Nested_Diagonal_Dynamic_NBody_Nonrigid_Shear_Activation_Audit_FAST"
    ),
    "upstream": UPSTREAM,
    "open_refs": G282_BLOCKED_OPEN_REFS,
    "levels": {
        "LEVEL1": LEVEL1,
        "LEVEL2": LEVEL2,
        "LEVEL3": LEVEL3,
        "LEVEL4": LEVEL4,
    },
    "nested": {
        "forward_reverse_pass": G282_NESTED_FORWARD_REVERSE_PASS,
        "hierarchical_position_reconstruction_pass":
            G282_HIERARCHICAL_POSITION_RECONSTRUCTION_PASS,
        "local_global_acceleration_equivalence_pass":
            G282_LOCAL_GLOBAL_ACCELERATION_EQUIVALENCE_PASS,
    },
    "nbody": {
        "generic_force_law_materialized": G282_GENERIC_NBODY_FORCE_LAW_MATERIALIZED,
        "internal_force_balance_pass": G282_NBODY_INTERNAL_FORCE_BALANCE_PASS,
        "translation_covariance_pass": G282_NBODY_TRANSLATION_COVARIANCE_PASS,
        "rotation_covariance_pass": G282_NBODY_ROTATION_COVARIANCE_PASS,
        "two_body_energy_drift": E2,
        "two_body_L_drift": L2,
        "three_body_energy_drift": E3,
        "three_body_L_drift": L3,
        "three_body_shape_error": shape3,
    },
    "shear": {
        "exact_3D_identity_pass": G282_3D_SHEAR_IDENTITY_EXACT_PASS,
        "isotropic_zero_pass": G282_ISOTROPIC_DIAGONAL_SHEAR_ZERO_PASS,
        "nonrigid_3D_shear_activated": G282_NONRIGID_DIAGONAL_3D_SHEAR_ACTIVATED,
        "shear_norm2_numeric": shear_norm2,
        "covariant_sigma_identification_proven":
            G282_COVARIANT_SIGMA_IDENTIFICATION_PROVEN,
    },
    "locks": {
        "physical_diagonal_field_EOM_derived":
            G282_PHYSICAL_DIAGONAL_FIELD_EOM_DERIVED,
        "new_GVH_Nbody_force_law_derived":
            G282_NEW_GVH_NBODY_FORCE_LAW_DERIVED,
        "Mercury_GVH_precession_prediction_authorized":
            G282_MERCURY_GVH_PRECESSION_PREDICTION_AUTHORIZED,
        "new_GVH_physics_validated":
            G282_NEW_GVH_PHYSICS_VALIDATED,
        "real_data_ready": G282_REAL_DATA_READY,
    },
    "verdict": {
        "G282_LOCAL_AUDIT_PASS": G282_LOCAL_AUDIT_PASS,
        "obstructions": G282_OBSTRUCTIONS,
        "next_authorized": G282_NEXT_AUTHORIZED,
    },
}

export_dir = Path("/content/gvh_exports") if Path("/content").exists() else Path("/mnt/data")
export_dir.mkdir(parents=True, exist_ok=True)
artifact_path = export_dir / (
    "gvh_0.3.2.7.3.7.3.3.28.2_"
    "Nested_Diagonal_Dynamic_NBody_Nonrigid_Shear_Activation_Audit_FAST.json"
)
artifact_path.write_text(
    json.dumps(artifact, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("LEVEL1 =", LEVEL1)
print("LEVEL2 =", LEVEL2)
print("LEVEL3 =", LEVEL3)
print("LEVEL4 =", LEVEL4)
print("G282_FULL_NESTED_FRAME_NBODY_ARCHITECTURE_MATERIALIZED =",
      G282_FULL_NESTED_FRAME_NBODY_ARCHITECTURE_MATERIALIZED)
print("G282_REFERENCE_PHYSICS_BENCHMARKS_PASS =",
      G282_REFERENCE_PHYSICS_BENCHMARKS_PASS)
print("G282_NONRIGID_DIAGONAL_3D_SHEAR_ACTIVATED =",
      G282_NONRIGID_DIAGONAL_3D_SHEAR_ACTIVATED)
print("G282_COVARIANT_SIGMA_IDENTIFICATION_PROVEN =",
      G282_COVARIANT_SIGMA_IDENTIFICATION_PROVEN)
print("G282_PHYSICAL_DIAGONAL_FIELD_EOM_DERIVED =",
      G282_PHYSICAL_DIAGONAL_FIELD_EOM_DERIVED)
print("G282_MERCURY_GVH_PRECESSION_PREDICTION_AUTHORIZED =",
      G282_MERCURY_GVH_PRECESSION_PREDICTION_AUTHORIZED)
print("G282_NEW_GVH_PHYSICS_VALIDATED =", G282_NEW_GVH_PHYSICS_VALIDATED)
print("G282_LOCAL_AUDIT_PASS =", G282_LOCAL_AUDIT_PASS)
print("G282_OBSTRUCTIONS =", G282_OBSTRUCTIONS)
print("G282_NEXT_AUTHORIZED =", G282_NEXT_AUTHORIZED)
print("G282 artifact =", artifact_path)

LEVEL1 = NESTED_DIAGONAL_NBODY_KINEMATICS_AND_REVERSE_DYNAMICS_MATERIALIZED
LEVEL2 = NEWTONIAN_NBODY_COVARIANCE_KEPLER_AND_LAGRANGE_BENCHMARKS_PASS
LEVEL3 = NUMERIC_NESTED_RECONSTRUCTION_CONSERVATION_AND_SHEAR_WITNESSES
LEVEL4 = BLOCKED_PHYSICAL_DIAGONAL_FIELD_EOM_AND_COVARIANT_SHEAR_IDENTIFICATION
G282_FULL_NESTED_FRAME_NBODY_ARCHITECTURE_MATERIALIZED = True
G282_REFERENCE_PHYSICS_BENCHMARKS_PASS = True
G282_NONRIGID_DIAGONAL_3D_SHEAR_ACTIVATED = True
G282_COVARIANT_SIGMA_IDENTIFICATION_PROVEN = False
G282_PHYSICAL_DIAGONAL_FIELD_EOM_DERIVED = False
G282_MERCURY_GVH_PRECESSION_PREDICTION_AUTHORIZED = False
G282_NEW_GVH_PHYSICS_VALIDATED = False
G282_LOCAL_AUDIT_PASS = True
G282_OBSTRUCTIONS = ['S_ij_IS_ONLY_A_3D_KINEMATIC_SHEAR_WITNESS_NOT_YET_COVARIANT_SIGMA_mn', 'D_k_OF_t_REMAINS_PRESCRIBED_FRAME_DATA_NOT_ACTION_DERIVED', 'NO_NEW_GVH_NBODY_FORCE_LAW_DERIVED', 'CUBIC_METRIC_STRESS_STILL_NOT_FULLY_COMPONENT_EXPANDED_IN_ARBITRARY_4D', 'TRIAD_OWN_KINETIC_TERM_STILL_ABSENT', 'MERCURY_GVH